# Laboratório 3 - Visão Computacional

> **Relatório completo:** a nossa execução (imagens dos resultados, matrizes estimadas e mosaicos) e a análise e discussão estão detalhadas no post:
>
> <https://kaykyb.github.io/ufabc-cv/posts/lab3/>
>
> Este notebook reúne os programas OpenCV do laboratório. A execução comentada dos experimentos e a análise dos resultados ficam no post acima.

## Parte 2: SIFT - Homografia - Panorama

(A) Elaborar um programa OpenCV. Para cada experimento, tira-se duas fotos com duas webcams: a primeira webcam captura `imagem1.jpg` de um plano e a segunda captura `imagem2.jpg` rotacionando ligeiramente a câmera para o lado, garantindo sobreposição de pelo menos 50%. Fizemos isso com uma pessoa (A.1) e com um livro (A.2).

Programa de captura das duas webcams (mostra os dois fluxos e salva `imagem1.jpg` e `imagem2.jpg` ao pressionar `x`):

In [ ]:
import cv2 as cv

# Abre as duas webcams
cap1 = cv.VideoCapture(0)
cap2 = cv.VideoCapture(1)

if not cap1.isOpened():
    print("Não foi possível abrir a câmera 0")
    exit()

if not cap2.isOpened():
    print("Não foi possível abrir a câmera 1")
    cap1.release()
    exit()

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1:
        print("Erro ao capturar da câmera 0")
        break

    if not ret2:
        print("Erro ao capturar da câmera 1")
        break

    # Exibe as imagens
    cv.imshow('Camera 0', frame1)
    cv.imshow('Camera 1', frame2)

    key = cv.waitKey(1) & 0xFF

    # Sai ao pressionar q
    if key == ord('q'):
        break

    # Salva uma imagem de cada câmera ao pressionar x
    if key == ord('x'):
        ok1, img1 = cv.imencode(".jpg", frame1)
        ok2, img2 = cv.imencode(".jpg", frame2)

        if ok1:
            with open("imagem1.jpg", "wb") as f:
                f.write(img1.tobytes())

        if ok2:
            with open("imagem2.jpg", "wb") as f:
                f.write(img2.tobytes())

        print("Imagens salvas: imagem1.jpg e imagem2.jpg")

# Libera recursos
cap1.release()
cap2.release()
cv.destroyAllWindows()


Dependências do pipeline de alinhamento e mosaico:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

Carregamos o par de imagens capturado e convertemos para escala de cinza, formato esperado pelo SIFT. Troque os caminhos para `images/livro1.jpg` e `images/livro2.jpg` para rodar o cenário do livro:

In [ ]:
path_img1 = 'images/pessoa1.jpg'
path_img2 = 'images/pessoa2.jpg'

img1 = cv2.imread(path_img1)
img2 = cv2.imread(path_img2)

if img1 is None or img2 is None:
    raise FileNotFoundError("Verifique se as imagens próprias foram salvas corretamente no diretório.")

# Conversão para escala de cinza para os extratores de feições
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

Detectamos as características com SIFT, pareamos por força bruta e aplicamos o teste de razão de Lowe para descartar correspondências ambíguas:

In [ ]:
# Utilizaremos o SIFT (Scale-Invariant Feature Transform)
sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

# Força bruta para encontrar os melhores matches usando a razão de Lowe
bf = cv2.BFMatcher()
matches_brutos = bf.knnMatch(des1, des2, k=2)

# Aplicação do teste de razão de Lowe para filtrar matches muito ambíguos
bons_matches = []
for m, n in matches_brutos:
    if m.distance < 0.75 * n.distance:
        bons_matches.append(m)

# Extração das coordenadas numéricas dos pontos correspondentes
src_pts = np.float32([kp1[m.queryIdx].pt for m in bons_matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in bons_matches]).reshape(-1, 1, 2)

print(f"Total de pontos correspondentes validados pré-geometria: {len(bons_matches)}")

Estimamos a homografia por **mínimos quadrados clássicos** (`method=0`, que ajusta todos os pontos de uma vez):

In [ ]:
# O método clássico tenta ajustar todos os pontos de uma vez (method=0)
H_ls, status_ls = cv2.findHomography(src_pts, dst_pts, method=0)

print("\n--- Matriz de Homografia via Mínimos Quadrados Clássicos ---")
print(H_ls)

E por **RANSAC** (robusto, com limiar de reprojeção de 5 px), contando inliers e outliers:

In [ ]:
# Definimos um limiar de reprojeção de 5 pixels para considerar um inlier
reproj_threshold = 5.0
H_ransac, mask_ransac = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, reproj_threshold)

print("\n--- Matriz de Homografia via RANSAC ---")
print(H_ransac)

# Contagem de Inliers e Outliers
inliers_count = np.sum(mask_ransac)
outliers_count = len(mask_ransac) - inliers_count
print(f"\nResultado RANSAC: {inliers_count} Inliers | {outliers_count} Outliers")

Visualizamos as correspondências aceitas pelo RANSAC (inliers, em verde):

In [ ]:
img_matches = cv2.drawMatches(img1, kp1, img2, kp2, bons_matches, None,
                              matchColor=(0, 255, 0), singlePointColor=None,
                              matchesMask=mask_ransac.ravel().tolist(), flags=2)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
plt.title("Correspondências (Inliers) Selecionadas pelo RANSAC")
plt.axis('off')
plt.show()

Por fim, projetamos a imagem 1 no plano da imagem 2 com `warpPerspective` e montamos o mosaico, comparando o resultado dos dois métodos:

In [ ]:
# Vamos rotacionar e projetar a Imagem 1 no plano da Imagem 2
width = img1.shape[1] + img2.shape[1]
height = max(img1.shape[0], img2.shape[0])

# Projeção usando Homografia por Mínimos Quadrados e por RANSAC
img1_warped_ls = cv2.warpPerspective(img1, H_ls, (width, height))
img1_warped_ransac = cv2.warpPerspective(img1, H_ransac, (width, height))

# Criação do Mosaico (colando a imagem 2 por cima da zona de projeção)
mosaico_ls = img1_warped_ls.copy()
mosaico_ls[0:img2.shape[0], 0:img2.shape[1]] = img2

mosaico_ransac = img1_warped_ransac.copy()
mosaico_ransac[0:img2.shape[0], 0:img2.shape[1]] = img2

# Exibição dos resultados comparativos
fig, ax = plt.subplots(2, 1, figsize=(14, 10))
ax[0].imshow(cv2.cvtColor(mosaico_ls, cv2.COLOR_BGR2RGB))
ax[0].set_title("Mosaico Final - Mínimos Quadrados Tradicionais (Pode falhar se houver outliers)")
ax[0].axis('off')

ax[1].imshow(cv2.cvtColor(mosaico_ransac, cv2.COLOR_BGR2RGB))
ax[1].set_title("Mosaico Final - RANSAC Robusto")
ax[1].axis('off')

plt.tight_layout()
plt.show()

(B) Elaborar outro programa OpenCV, modificando o código acima, para fazer a leitura de duas webcams pelo mesmo código, montando e mostrando o mosaico numa janela de vídeo a cada leitura das webcams:

In [ ]:
import cv2
import numpy as np

# Inicializa as webcams
cap1 = cv2.VideoCapture(0)
cap2 = cv2.VideoCapture(1)

# Inicializa o SIFT e o Matcher de Força Bruta
sift = cv2.SIFT_create()
bf = cv2.BFMatcher()

print("Processando Mosaico em tempo real... Pressione 'q' para SAIR.")

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    # Conversão para escala de cinza
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Detecção de características e cômputo dos descritores
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    # Garantia de que encontramos descritores em ambas as imagens
    if des1 is not None and des2 is not None and len(des1) > 0 and len(des2) > 0:
        matches_brutos = bf.knnMatch(des1, des2, k=2)

        # Filtro de Lowe
        bons_matches = []

        for match in matches_brutos:
            if len(match) == 2:
                m, n = match

                if m.distance < 0.75 * n.distance:
                    bons_matches.append(m)

        # São necessários pelo menos 4 pontos para calcular a matriz de Homografia
        if len(bons_matches) >= 4:
            src_pts = np.float32(
                [kp1[m.queryIdx].pt for m in bons_matches]
            ).reshape(-1, 1, 2)

            dst_pts = np.float32(
                [kp2[m.trainIdx].pt for m in bons_matches]
            ).reshape(-1, 1, 2)

            # Usamos apenas o RANSAC para garantir robustez e performance no vídeo
            H_ransac, mask_ransac = cv2.findHomography(
                src_pts,
                dst_pts,
                cv2.RANSAC,
                5.0
            )

            if H_ransac is not None:
                width = frame1.shape[1] + frame2.shape[1]
                height = max(frame1.shape[0], frame2.shape[0])

                # Projeção e Costura
                mosaico_ransac = cv2.warpPerspective(
                    frame1,
                    H_ransac,
                    (width, height)
                )

                mosaico_ransac[
                    0:frame2.shape[0],
                    0:frame2.shape[1]
                ] = frame2

                cv2.imshow(
                    "Mosaico em Tempo Real (RANSAC)",
                    mosaico_ransac
                )
            else:
                # Fallback se a homografia falhar
                cv2.imshow(
                    "2Mosaico em Tempo Real (RANSAC)",
                    frame2
                )
        else:
            # Mostra apenas a câmera 2 se não houver matches suficientes
            cv2.imshow(
                "3Mosaico em Tempo Real (RANSAC)",
                frame2
            )
    else:
        # Caso não haja descritores válidos
        cv2.imshow(
            "4Mosaico em Tempo Real (RANSAC)",
            frame2
        )

    # Pressione 'q' para encerrar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap1.release()
cap2.release()
cv2.destroyAllWindows()
